In [2]:
import numpy as np
import tensorflow as tf

In [3]:
# utwórzmy tymczasową zmienną npz, w której będziemy przechowywać każdy z trzech zestawów danych Audiobooków
npz = np.load('Audiobooks_data_train.npz')

# wyodrębniamy dane wejściowe za pomocą słowa kluczowego, pod którym je zapisaliśmy
# aby mieć pewność, że wszystkie są liczbami zmiennoprzecinkowymi, zajmijmy się również tym
train_inputs = npz['inputs'].astype(np.float32)
# cele muszą być typu int ze względu na funkcję celu sparse_categorical_crossentropy
train_targets = npz['targets'].astype(np.int32)

# ładujemy dane walidacyjne do zmiennej tymczasowej
npz = np.load('Audiobooks_data_validation.npz')
# możemy załadować dane wejściowe i targety w tym samym wierszu
validation_inputs, validation_targets = npz['inputs'].astype(np.float32), npz['targets'].astype(np.int32)

# ładujemy dane testowe do zmiennej tymczasowej
npz = np.load('Audiobooks_data_test.npz')
# tworzymy 2 zmienne, które będą zawierać dane wejściowe testu i cele testu
test_inputs, test_targets = npz['inputs'].astype(np.float32), npz['targets'].astype(np.int32)

In [4]:
# Ustaw rozmiary wejściowe i wyjściowe
input_size = 10
output_size = 2
# Użyj tego samego rozmiaru ukrytej warstwy dla obu ukrytych warstw. Nie jest to konieczne.
hidden_layer_size_1 = 64
hidden_layer_size_2 = 32

model = tf.keras.Sequential([
    tf.keras.layers.Dense(hidden_layer_size_1, activation='relu',
                          kernel_initializer='he_normal'),
    tf.keras.layers.Dense(hidden_layer_size_2, activation='relu',
                          kernel_initializer='he_normal'),
    tf.keras.layers.Dense(output_size, activation='softmax')
])

# Wybieramy funkcję optymalizacji i funkcję straty
# oraz metryki, które chcemy uzyskać w każdej iteracji
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [5]:
# ustawiamy rozmiar podawanej partii danych
batch_size = 32

# ustawiamy maksymalną liczbę epok treningowych
max_epochs = 200

# ustaw mechanizm wczesnego zatrzymywania, który ochroni nasz model przed przetrenowaniem
# ustawmy patience=2, aby być nieco tolerancyjnym na losowe wzrosty strat walidacji
early_stopping = tf.keras.callbacks.EarlyStopping(
    patience=8,
    restore_best_weights=True
)

# i nauczmy model
model.fit(train_inputs, # dane wejściowe trenujące
          train_targets, # targety wejściowe trenujące
          batch_size=batch_size, # rozmiar podawanej partii
          epochs=max_epochs, # maksymalna ilość epok gdyby wczesne zatrzymanie nie zadziałało
          # callbacki to funkcje wywoływane przez zadanie po jego zakończeniu
          # zadanie tutaj polega na sprawdzeniu, czy val_loss rośnie
          callbacks=[early_stopping], # mechanizm "early stopping" - zapobieganie przetrenowaniu
          validation_data=(validation_inputs, validation_targets), # dane walidacyjne
          verbose = 1 # sposób pokazania treningu modelu
          )

Epoch 1/200
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8117 - loss: 0.4654 - val_accuracy: 0.8837 - val_loss: 0.3384
Epoch 2/200
112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8815 - loss: 0.3158 - val_accuracy: 0.8971 - val_loss: 0.2990
Epoch 3/200
112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8894 - loss: 0.2899 - val_accuracy: 0.8993 - val_loss: 0.2826
Epoch 4/200
112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8941 - loss: 0.2765 - val_accuracy: 0.9060 - val_loss: 0.2726
Epoch 5/200
112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8969 - loss: 0.2681 - val_accuracy: 0.9083 - val_loss: 0.2660
Epoch 6/200
112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8983 - loss: 0.2621 - val_accuracy: 0.9128 - val_loss: 0.2609
Epoch 7/200
112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8991 - loss: 0.2575 - val_accuracy: 0.9105 - val_loss: 0.2560
Epoch 8/200
112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9005 - loss: 0.2534 - val_accu

In [6]:
test_loss, test_accuracy = model.evaluate(test_inputs, test_targets)
print('\nTest loss: {0:.2f}. Test accuracy: {1:.2f}%'.format(test_loss, test_accuracy*100.))

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9308 - loss: 0.2261 

Test loss: 0.23. Test accuracy: 93.08%
